# Fit Pink Trombone params to a real speech clip

Gradient descent on the 13-dim parameter trajectory of `pink_trombone_ola` (the faster IIR/OLA FIR synth) to approximate `examples/this-is-samuel.wav`.

Loss: multi-scale log-magnitude STFT L1 (phase-invariant — waveform MSE won't work because glottal pulses won't phase-align with the target).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from scipy.io import wavfile
from scipy.signal import resample_poly
import plotly.graph_objects as go
from IPython.display import Audio, display

from samuel.pink_trombone import (
    pink_trombone_ola,
    PARAM_NAMES,
    N_PARAMS,
    SAMPLE_RATE,
    SAMPLES_PER_FRAME,
    CONTROL_RATE,
)

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Load and resample target audio

In [ ]:
WAV_PATH = Path("./examples/this-is-samuel.wav").resolve()
sr_in, wav_int16 = wavfile.read(WAV_PATH)
print(f"loaded: sr={sr_in} Hz, samples={wav_int16.shape[0]} ({wav_int16.shape[0]/sr_in:.2f} s), dtype={wav_int16.dtype}")

# Mono float32, normalized to [-1, 1]
if wav_int16.ndim > 1:
    wav_int16 = wav_int16.mean(axis=1)
wav = wav_int16.astype(np.float32) / 32768.0

# Resample 48000 -> 44100 via rational factor 441/480
from math import gcd
g = gcd(sr_in, SAMPLE_RATE)
up, down = SAMPLE_RATE // g, sr_in // g
wav_rs = resample_poly(wav, up, down).astype(np.float32)
print(f"resampled: {sr_in} -> {SAMPLE_RATE} Hz (poly {up}/{down}), samples={wav_rs.shape[0]}")

# Truncate to a whole number of control frames
T = wav_rs.shape[0] // SAMPLES_PER_FRAME
T_samples = T * SAMPLES_PER_FRAME
wav_rs = wav_rs[:T_samples]
target = torch.from_numpy(wav_rs).unsqueeze(0).to(device)
print(f"target: shape={tuple(target.shape)}, T={T} frames ({T/CONTROL_RATE:.2f} s)")

print("\nTarget:")
display(Audio(target[0].cpu().numpy(), rate=SAMPLE_RATE))

## 2. Constrained parameterization

Trainable raw scalars for 9 params; vibrato + tractLength are frozen. `to_synth_params` maps raw → constrained [B, T, 13] via sigmoid + affine.

In [ ]:
# (lo, hi, init) per param. None entries are frozen at their default.
PARAM_SPEC = {
    "noise":                (0.0, 0.5, 0.1),
    "frequency":            (80.0, 400.0, 140.0),
    "tenseness":            (0.0, 1.0, 0.6),
    "intensity":            (0.0, 1.0, 1.0),
    "loudness":             (0.0, 1.0, 1.0),
    "tongueIndex":          (10.0, 35.0, 20.0),
    "tongueDiameter":       (1.5, 3.5, 2.4),
    "vibratoWobble":        None,   # frozen at 0
    "vibratoFrequency":     None,   # frozen at 6
    "vibratoGain":          None,   # frozen at 0
    "tractLength":          None,   # frozen at 44
    "constrictionIndex":    (0.0, 44.0, 30.0),
    "constrictionDiameter": (-0.5, 3.0, 3.0),
}
FROZEN_VALUES = {
    "vibratoWobble": 0.0,
    "vibratoFrequency": 6.0,
    "vibratoGain": 0.0,
    "tractLength": 44.0,
}

trainable_names = [n for n in PARAM_NAMES if PARAM_SPEC[n] is not None]
n_trainable = len(trainable_names)
print(f"trainable params: {n_trainable} x {T} frames = {n_trainable * T} scalars")

# Precompute per-param (lo, hi) tensors aligned to trainable_names
_lo = torch.tensor([PARAM_SPEC[n][0] for n in trainable_names], device=device)
_hi = torch.tensor([PARAM_SPEC[n][1] for n in trainable_names], device=device)
_init = torch.tensor([PARAM_SPEC[n][2] for n in trainable_names], device=device)

# Inverse-sigmoid of normalized init so sigmoid(raw_init) * (hi-lo) + lo == init
_init_norm = ((_init - _lo) / (_hi - _lo)).clamp(1e-4, 1 - 1e-4)
_raw_init = torch.log(_init_norm / (1 - _init_norm))  # [n_trainable]

raw = _raw_init.view(1, 1, n_trainable).expand(1, T, n_trainable).contiguous().clone()
raw.requires_grad_(True)

def to_synth_params(raw: torch.Tensor) -> torch.Tensor:
    """raw [1, T, n_trainable] -> synth params [1, T, 13] in valid ranges."""
    constrained = torch.sigmoid(raw) * (_hi - _lo) + _lo  # [1, T, n_trainable]
    out = torch.zeros(1, T, N_PARAMS, device=raw.device, dtype=raw.dtype)
    for j, name in enumerate(trainable_names):
        i = PARAM_NAMES.index(name)
        out[..., i] = constrained[..., j]
    for name, val in FROZEN_VALUES.items():
        i = PARAM_NAMES.index(name)
        out[..., i] = val
    return out

# Sanity check: initial synth params should equal the init table
with torch.no_grad():
    p0 = to_synth_params(raw)[0, 0]
    for name in PARAM_NAMES:
        print(f"  {name:22s} = {p0[PARAM_NAMES.index(name)].item():.3f}")

## 3. Multi-scale STFT loss

In [ ]:
_WINDOWS = {n: torch.hann_window(n, device=device) for n in (512, 1024, 2048)}

def multi_stft_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    loss = pred.new_zeros(())
    for n_fft, window in _WINDOWS.items():
        hop = n_fft // 4
        s_p = torch.stft(pred, n_fft, hop, window=window, return_complex=True).abs()
        s_t = torch.stft(target, n_fft, hop, window=window, return_complex=True).abs()
        loss = loss + (torch.log1p(s_p) - torch.log1p(s_t)).abs().mean()
    return loss / len(_WINDOWS)

# Sanity: initial loss
with torch.no_grad():
    init_pred = pink_trombone_ola(to_synth_params(raw), seed=0)
    print(f"initial loss: {multi_stft_loss(init_pred, target).item():.4f}")
    print(f"initial pred shape: {tuple(init_pred.shape)}, target shape: {tuple(target.shape)}")

## 4. Training loop

In [ ]:
import time
from tqdm.auto import tqdm

def _norm_np(x: torch.Tensor) -> np.ndarray:
    a = x.detach().cpu().numpy()
    p = np.abs(a).max()
    return a / p * 0.9 if p > 1e-6 else a

def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

N_STEPS = 200
LR = 0.05
IR_LENGTH = 256  # main cost driver: sequential graph depth inside the synth
optimizer = torch.optim.Adam([raw], lr=LR)

losses: list[float] = []
best_loss = float("inf")
best_pred = None
best_params = None

# Per-phase time totals (seconds)
phase_t = {"fwd": 0.0, "loss": 0.0, "bwd": 0.0, "opt": 0.0, "display": 0.0}

# Live audio widget — one handle, updated in-place each step
audio_h = display(
    Audio(np.zeros(SAMPLES_PER_FRAME, dtype=np.float32), rate=SAMPLE_RATE),
    display_id=True,
)

pbar = tqdm(range(N_STEPS), desc="fitting")
for step in pbar:
    _sync(); t0 = time.perf_counter()
    optimizer.zero_grad()
    synth_params = to_synth_params(raw)
    pred = pink_trombone_ola(synth_params, seed=0, ir_length=IR_LENGTH)
    assert pred.shape == target.shape, (pred.shape, target.shape)
    _sync(); t1 = time.perf_counter()

    loss = multi_stft_loss(pred, target)
    _sync(); t2 = time.perf_counter()

    loss.backward()
    _sync(); t3 = time.perf_counter()

    optimizer.step()
    _sync(); t4 = time.perf_counter()

    audio_h.update(Audio(_norm_np(pred[0]), rate=SAMPLE_RATE))
    t5 = time.perf_counter()

    phase_t["fwd"]     += t1 - t0
    phase_t["loss"]    += t2 - t1
    phase_t["bwd"]     += t3 - t2
    phase_t["opt"]     += t4 - t3
    phase_t["display"] += t5 - t4

    losses.append(loss.item())
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_pred = pred.detach().clone()
        best_params = synth_params.detach().clone()

    pbar.set_postfix(
        loss=f"{loss.item():.4f}",
        best=f"{best_loss:.4f}",
        step_s=f"{t4 - t0:.2f}",
    )

print(f"\nbest loss = {best_loss:.4f}\n")
total = sum(phase_t.values())
print(f"{'phase':<10} {'total (s)':>10} {'ms/step':>10} {'%':>6}")
for k, v in phase_t.items():
    print(f"{k:<10} {v:10.2f} {v / N_STEPS * 1000:10.1f} {v / total * 100:5.1f}%")
print(f"{'TOTAL':<10} {total:10.2f} {total / N_STEPS * 1000:10.1f} {100.0:5.1f}%")

## 5. Loss curve (plotly)

In [ ]:
fig = go.Figure(
    data=[go.Scatter(y=losses, mode="lines", name="multi-STFT L1")],
    layout=dict(
        title="Fitting loss vs. optimization step",
        xaxis_title="step",
        yaxis_title="loss (log scale)",
        yaxis_type="log",
        template="plotly_white",
        width=800,
        height=400,
    ),
)
fig.show()

## 6. Listen

In [ ]:
def norm(x: torch.Tensor) -> np.ndarray:
    x = x.cpu().numpy()
    peak = np.abs(x).max()
    return x / peak * 0.9 if peak > 1e-6 else x

print("Target:")
display(Audio(norm(target[0]), rate=SAMPLE_RATE))

print("Fitted (best step):")
display(Audio(norm(best_pred[0]), rate=SAMPLE_RATE))

## 7. Fitted parameter trajectories

In [ ]:
show_names = ["frequency", "tenseness", "tongueIndex", "tongueDiameter", "constrictionIndex", "constrictionDiameter"]
frames = np.arange(T) / CONTROL_RATE  # seconds

fig = go.Figure()
for name in show_names:
    i = PARAM_NAMES.index(name)
    fig.add_trace(go.Scatter(x=frames, y=best_params[0, :, i].cpu().numpy(), mode="lines+markers", name=name))
fig.update_layout(
    title="Fitted parameter trajectories (best step)",
    xaxis_title="time (s)",
    yaxis_title="value",
    template="plotly_white",
    width=900,
    height=450,
)
fig.show()

## 8. Spectrum comparison (target vs. fitted OLA)

In [ ]:
import librosa
from plotly.subplots import make_subplots

N_FFT = 2048
HOP = N_FFT // 4
N_MELS = 80
F_MIN, F_MAX = 0.0, 2000.0
_win = torch.hann_window(N_FFT, device=device)

_mel_fb = torch.from_numpy(
    librosa.filters.mel(sr=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX)
).to(device)  # [N_MELS, F]

mel_centers = librosa.mel_frequencies(n_mels=N_MELS, fmin=F_MIN, fmax=F_MAX)

def _logmel(x: torch.Tensor) -> np.ndarray:
    """[T_samples] -> log-mel spectrogram [N_MELS, N_frames] on CPU."""
    s = torch.stft(x.unsqueeze(0), N_FFT, HOP, window=_win, return_complex=True).abs()  # [1, F, N]
    m = _mel_fb @ s[0]  # [N_MELS, N]
    return torch.log1p(m).cpu().numpy()

mel_target = _logmel(target[0])
mel_fit    = _logmel(best_pred[0])

times = np.arange(mel_target.shape[1]) * HOP / SAMPLE_RATE
zmax = float(max(mel_target.max(), mel_fit.max()))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("target", "fitted (OLA)"))
for col, spec in [(1, mel_target), (2, mel_fit)]:
    fig.add_trace(go.Heatmap(z=spec, x=times, y=mel_centers, zmin=0, zmax=zmax,
                             colorscale="Viridis", showscale=(col == 2)),
                  row=1, col=col)
fig.update_xaxes(title_text="time (s)")
fig.update_yaxes(title_text="mel center freq (Hz)", row=1, col=1)
fig.update_layout(title=f"log-mel spectrogram (n_mels={N_MELS}, n_fft={N_FFT})",
                  template="plotly_white", width=1000, height=400)
fig.show()

# Time-averaged mel spectra for a direct line-plot comparison
avg_target = mel_target.mean(axis=1)
avg_fit    = mel_fit.mean(axis=1)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_target, name="target",       mode="lines"))
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_fit,    name="fitted (OLA)", mode="lines"))
fig2.update_layout(
    title="Time-averaged log-mel spectrum",
    xaxis_title="mel center freq (Hz)", yaxis_title="log(1 + mel energy)",
    template="plotly_white", width=900, height=400,
)
fig2.show()

## 9. Resynth with exact (non-OLA) waveguide

Fitting used `pink_trombone_ola` (OLA FIR approximation). Here we resynthesize the same best params through the exact sequential Kelly–Lochbaum waveguide (`pink_trombone`) to see how much the OLA approximation is shaping the result.

In [ ]:
from samuel.pink_trombone import pink_trombone

with torch.no_grad():
    pred_exact = pink_trombone(best_params, seed=0)
    loss_ola   = multi_stft_loss(best_pred,  target).item()
    loss_exact = multi_stft_loss(pred_exact, target).item()

print(f"multi-STFT loss vs. target:")
print(f"  OLA   (training synth): {loss_ola:.4f}")
print(f"  exact (sequential WG):  {loss_exact:.4f}")

rms_diff = (pred_exact - best_pred).pow(2).mean().sqrt().item()
rms_exact = pred_exact.pow(2).mean().sqrt().item()
print(f"  RMS(exact - OLA) = {rms_diff:.4f}   (relative to exact RMS {rms_exact:.4f}: {rms_diff/rms_exact:.2%})")

print("\nTarget:")
display(Audio(norm(target[0]),     rate=SAMPLE_RATE))
print("Fitted — OLA FIR (what the optimizer saw):")
display(Audio(norm(best_pred[0]),  rate=SAMPLE_RATE))
print("Fitted — exact sequential waveguide:")
display(Audio(norm(pred_exact[0]), rate=SAMPLE_RATE))

# Log-mel spectrograms: exact vs OLA (same params)
mel_exact = _logmel(pred_exact[0])
zmax2 = float(max(mel_fit.max(), mel_exact.max()))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("fitted (OLA)", "fitted (exact)"))
for col, spec in [(1, mel_fit), (2, mel_exact)]:
    fig.add_trace(go.Heatmap(z=spec, x=times, y=mel_centers, zmin=0, zmax=zmax2,
                             colorscale="Viridis", showscale=(col == 2)),
                  row=1, col=col)
fig.update_xaxes(title_text="time (s)")
fig.update_yaxes(title_text="mel center freq (Hz)", row=1, col=1)
fig.update_layout(title="OLA vs. exact waveguide (same fitted params) — log-mel",
                  template="plotly_white", width=1000, height=400)
fig.show()

avg_exact = mel_exact.mean(axis=1)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_target, name="target",         mode="lines"))
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_fit,    name="fitted (OLA)",   mode="lines"))
fig2.add_trace(go.Scatter(x=mel_centers, y=avg_exact,  name="fitted (exact)", mode="lines"))
fig2.update_layout(
    title="Time-averaged log-mel spectrum — target vs OLA vs exact",
    xaxis_title="mel center freq (Hz)", yaxis_title="log(1 + mel energy)",
    template="plotly_white", width=900, height=400,
)
fig2.show()

## 10. Pitch (f0) comparison

Estimated with `librosa.pyin` (probabilistic YIN) — robust to voiced/unvoiced transitions, NaN on unvoiced frames. Compared against the commanded `frequency` parameter the optimizer settled on.

In [ ]:
PITCH_FMIN, PITCH_FMAX = 70.0, 500.0
PITCH_HOP = 512  # ~11.6 ms at 44.1 kHz

def _pyin_f0(x: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    wav = x.detach().cpu().numpy().astype(np.float32)
    f0, _, _ = librosa.pyin(
        wav, sr=SAMPLE_RATE,
        fmin=PITCH_FMIN, fmax=PITCH_FMAX,
        frame_length=2048, hop_length=PITCH_HOP,
    )
    t = librosa.times_like(f0, sr=SAMPLE_RATE, hop_length=PITCH_HOP)
    return t, f0

t_pit, f0_target = _pyin_f0(target[0])
_,     f0_ola    = _pyin_f0(best_pred[0])
_,     f0_exact  = _pyin_f0(pred_exact[0])

freq_idx = PARAM_NAMES.index("frequency")
commanded = best_params[0, :, freq_idx].cpu().numpy()
t_cmd = np.arange(T) / CONTROL_RATE

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_pit, y=f0_target, name="target (pyin)",       mode="lines"))
fig.add_trace(go.Scatter(x=t_pit, y=f0_ola,    name="fitted OLA (pyin)",   mode="lines"))
fig.add_trace(go.Scatter(x=t_pit, y=f0_exact,  name="fitted exact (pyin)", mode="lines"))
fig.add_trace(go.Scatter(x=t_cmd, y=commanded, name="commanded frequency", mode="lines+markers",
                         line=dict(dash="dash")))
fig.update_layout(
    title="Pitch (f0) trajectories",
    xaxis_title="time (s)",
    yaxis_title="f0 (Hz)",
    yaxis_range=[PITCH_FMIN, PITCH_FMAX],
    template="plotly_white", width=1000, height=450,
)
fig.show()

# MAE in cents on frames where both target and each synth are voiced
def _cents_mae(a: np.ndarray, b: np.ndarray) -> float:
    m = np.isfinite(a) & np.isfinite(b)
    if not m.any():
        return float("nan")
    return float(np.mean(np.abs(1200.0 * np.log2(a[m] / b[m]))))

print(f"voiced-frame MAE vs. target:")
print(f"  fitted OLA:   {_cents_mae(f0_target, f0_ola):.1f} cents")
print(f"  fitted exact: {_cents_mae(f0_target, f0_exact):.1f} cents")